# Tiny CPU model workflow

Builds a **reduced-filter** 3D CNN (far smaller than the paper's `(128, 128, 50)` input / `(64, 64, 128, 256)` filters) and runs one bounded train -> evaluate -> save/reload flow on tiny synthetic arrays, CPU-only, in a few seconds. This is **not** a reproduction of the paper's results: it is a smoke-sized tour of the `multimodal_ad.models` API.

Requires the `model` extra: `uv sync --extra model`. All logic lives in `multimodal_ad.models` (see [`src/multimodal_ad/models/__init__.py`](../src/multimodal_ad/models/__init__.py)); this notebook only calls it and displays results.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from multimodal_ad.models.architecture import Cnn3DConfig, build_3d_cnn
from multimodal_ad.models.evaluation import evaluate_predictions
from multimodal_ad.models.training import TrainingConfig, load_model, save_model, train_model

## Tiny synthetic training arrays

Random noise, not real scans; only shapes/dtypes matter here.

In [ ]:
rng = np.random.default_rng(1234)
SIZE = 48  # far smaller than the paper's 128x128x50 input, CPU-friendly

x_train = rng.random((8, SIZE, SIZE, SIZE, 1)).astype("float32")
y_train = (np.arange(8) % 2).astype("float32")
x_val = rng.random((4, SIZE, SIZE, SIZE, 1)).astype("float32")
y_val = (np.arange(4) % 2).astype("float32")

## Build a reduced-filter 3D CNN

Same four-block topology as the paper's architecture, tiny filter counts.

In [ ]:
config = Cnn3DConfig(
    width=SIZE,
    height=SIZE,
    depth=SIZE,
    filters=(4, 4, 8, 8),  # paper default: (64, 64, 128, 256)
    dense_units=16,  # paper default: 512
    name="tiny-cnn",
)
model = build_3d_cnn(config)
model.summary()

## Train for one bounded epoch

`epochs=1` and no early-stopping baseline: this is a workflow demo, not a real training run.

In [ ]:
output_dir = Path(tempfile.mkdtemp(prefix="multimodal-ad-tiny-model-"))
training_config = TrainingConfig(
    epochs=1,
    batch_size=2,
    checkpoint_path=output_dir / "tiny.keras",
    early_stopping_baseline=None,
    verbose=0,
)
result = train_model(model, x_train, y_train, x_val, y_val, training_config)
result.val_loss, result.val_accuracy

## Correct evaluation metrics

Accuracy/sensitivity/specificity/AUC from thresholded probabilities (`multimodal_ad.models.evaluation`).

In [ ]:
probabilities = result.model.predict(x_val, verbose=0).reshape(-1)
metrics = evaluate_predictions(y_val, probabilities)
metrics

## Save, reload, and confirm identical predictions

In [ ]:
final_path = output_dir / "tiny-final.keras"
save_model(result.model, final_path)
reloaded_model = load_model(final_path)

bool(np.allclose(
    result.model.predict(x_val, verbose=0),
    reloaded_model.predict(x_val, verbose=0),
))

## Next steps

- See `03-explainability.ipynb` for Grad-CAM and AAL2 region ranking on top of a model like this one.
- `docs/reproducibility.md` documents exactly what is and isn't validated against the paper's published numbers.